In [ ]:
import numpy as np
import pandas as pd
import joblib
from flask import Flask, request, jsonify, render_template
from utils import validate_input

app = Flask(__name__)

# تحميل الموديل
model = joblib.load("../models/heart_model.pkl")

FEATURES_ORDER = [
    "age","sex","cp","trestbps","chol","fbs",
    "restecg","thalach","exang","oldpeak",
    "slope","ca","thal"
]

OPTIONS = {
    'sex': [('1', 'ذكر'), ('0', 'أنثى')],
    'cp': [('0', 'Typical'), ('1', 'Atypical'), ('2', 'Non-anginal'), ('3', 'Asymptomatic')],
    'fbs': [('1', 'نعم'), ('0', 'لا')],
    'restecg': [('0', 'Normal'), ('1', 'Abnormal'), ('2', 'LVH')],
    'exang': [('1', 'نعم'), ('0', 'لا')],
    'slope': [('0', 'Up'), ('1', 'Flat'), ('2', 'Down')],
    'thal': [('0', 'Normal'), ('1', 'Fixed'), ('2', 'Reversible')],
    'ca': [('0','0'),('1','1'),('2','2'),('3','3')]
}

# ======================
# الصفحة الرئيسية (UI)
# ======================
@app.route('/')
def home():
    return render_template("index.html", cols=FEATURES_ORDER, options=OPTIONS)

# ======================
# Prediction (UI)
# ======================
@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = {col: float(request.form.get(col, 0)) for col in FEATURES_ORDER}

        errors = validate_input(data)
        if errors:
            return render_template("index.html", cols=FEATURES_ORDER,
                                   options=OPTIONS,
                                   result="❌ " + " | ".join(errors))

        features = np.array([list(data.values())])

        pred = model.predict(features)[0]
        prob = model.predict_proba(features)[0][1]

        level = "Low"
        if prob > 0.7:
            level = "High"
        elif prob > 0.4:
            level = "Medium"

        result = f"Prediction: {pred} | Risk: {prob:.2f} | Level: {level}"

        return render_template("index.html", cols=FEATURES_ORDER,
                               options=OPTIONS,
                               result=result)

    except Exception as e:
        return f"Error: {e}"

# ======================
# API Prediction
# ======================
@app.route('/api/predict', methods=['POST'])
def api_predict():
    try:
        data = request.json

        ordered = {col: float(data.get(col, 0)) for col in FEATURES_ORDER}

        errors = validate_input(ordered)
        if errors:
            return jsonify({"errors": errors}), 400

        features = np.array([list(ordered.values())])

        pred = model.predict(features)[0]
        prob = model.predict_proba(features)[0][1]

        return jsonify({
            "prediction": int(pred),
            "risk": float(prob)
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# ======================
# CSV Batch
# ======================
@app.route('/batch', methods=['POST'])
def batch():
    try:
        file = request.files['file']
        df = pd.read_csv(file)

        if not all(col in df.columns for col in FEATURES_ORDER):
            return "❌ CSV غير مطابق"

        preds = model.predict(df[FEATURES_ORDER])
        probs = model.predict_proba(df[FEATURES_ORDER])[:, 1]

        df['Prediction'] = preds
        df['Risk'] = probs

        return df.to_html()

    except Exception as e:
        return f"Error: {e}"

# ======================
# تشغيل السيرفر
# ======================
if __name__ == '__main__':
    app.run(debug=True)